# 01 — Exploratory Data Analysis (EDA)
## AI-Powered Medical Image Analysis System

This notebook explores the chest X-ray dataset before training.

### Objectives:
- Understand class distribution
- Visualize sample images
- Check image dimensions
- Identify class imbalance

In [ ]:
import os
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import defaultdict

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

DATA_DIR = '../data/raw'
print('EDA initialized.')

## 1. Dataset Overview

In [ ]:
# Count images per split and class
stats = defaultdict(dict)

for split in ['train', 'val', 'test']:
    split_path = os.path.join(DATA_DIR, split)
    if not os.path.exists(split_path):
        continue
    for cls in os.listdir(split_path):
        cls_path = os.path.join(split_path, cls)
        if os.path.isdir(cls_path):
            count = len([f for f in os.listdir(cls_path)
                         if f.lower().endswith(('.jpg','.jpeg','.png'))])
            stats[split][cls] = count

print('Dataset Statistics:')
print('-' * 40)
for split, classes in stats.items():
    print(f'  {split.upper()}')
    for cls, count in classes.items():
        print(f'    {cls:<15}: {count:>5}')
print('-' * 40)

## 2. Class Distribution Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Class Distribution per Split', fontsize=14, fontweight='bold')

colors = {'NORMAL': '#4CAF50', 'PNEUMONIA': '#F44336'}

for i, (split, classes) in enumerate(stats.items()):
    labels = list(classes.keys())
    values = list(classes.values())
    bar_colors = [colors.get(l, '#2196F3') for l in labels]
    
    axes[i].bar(labels, values, color=bar_colors, edgecolor='white', linewidth=1.5)
    axes[i].set_title(f'{split.upper()} Split', fontsize=12)
    axes[i].set_ylabel('Image Count')
    for j, v in enumerate(values):
        axes[i].text(j, v + 5, str(v), ha='center', fontweight='bold')
    axes[i].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/plots/eda_class_distribution.png', dpi=150)
plt.show()
print('Class distribution plot saved.')

## 3. Sample Images

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Sample Chest X-Ray Images', fontsize=15, fontweight='bold')

for row_idx, cls in enumerate(['NORMAL', 'PNEUMONIA']):
    cls_path = os.path.join(DATA_DIR, 'train', cls)
    if not os.path.exists(cls_path):
        print(f'Path not found: {cls_path}')
        continue
    images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    
    for col_idx in range(4):
        if col_idx >= len(images): break
        img = Image.open(os.path.join(cls_path, images[col_idx])).convert('RGB')
        axes[row_idx][col_idx].imshow(img, cmap='gray')
        axes[row_idx][col_idx].set_title(
            cls, fontsize=10,
            color='green' if cls == 'NORMAL' else 'red'
        )
        axes[row_idx][col_idx].axis('off')

plt.tight_layout()
plt.savefig('../outputs/plots/eda_sample_images.png', dpi=150)
plt.show()

## 4. Image Size Distribution

In [ ]:
widths, heights = [], []
sample_path = os.path.join(DATA_DIR, 'train', 'NORMAL')

if os.path.exists(sample_path):
    files = [f for f in os.listdir(sample_path) if f.endswith(('.jpg','.jpeg','.png'))][:100]
    for f in files:
        img = Image.open(os.path.join(sample_path, f))
        widths.append(img.size[0])
        heights.append(img.size[1])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Image Size Distribution (NORMAL, n=100)', fontsize=13)
    
    axes[0].hist(widths, bins=20, color='#0066CC', edgecolor='white')
    axes[0].set_title('Width Distribution'); axes[0].set_xlabel('Pixels')
    
    axes[1].hist(heights, bins=20, color='#FF6600', edgecolor='white')
    axes[1].set_title('Height Distribution'); axes[1].set_xlabel('Pixels')
    
    plt.tight_layout()
    plt.show()
    print(f'Width  — Mean: {np.mean(widths):.0f}, Min: {min(widths)}, Max: {max(widths)}')
    print(f'Height — Mean: {np.mean(heights):.0f}, Min: {min(heights)}, Max: {max(heights)}')
else:
    print(f'Path not found: {sample_path}')
    print('Run: python scripts/generate_demo_data.py first')

## 5. Key EDA Findings

Based on the analysis above:

1. **Class Imbalance**: PNEUMONIA images outnumber NORMAL in training (~3:1)
   - Solution: Use class weights during training
   
2. **Image Size Variance**: Images have variable dimensions
   - Solution: Resize all to 224×224 during preprocessing

3. **Visual Differences**:
   - Normal: Clear lung fields, sharp costophrenic angles
   - Pneumonia: Opacities, consolidations, haziness

4. **Next Steps**: Preprocessing → Model Building → Training